# OpenShell UI on Google Colab

Colab の実行環境内でダッシュボードを起動し、ノートブック内に iframe で表示します。
ブラウザだけでデモを試せます。GPU ランタイムでも同じ手順で動きます。

**公開トンネルは意図的に開始しません。** Colab の FAQ は、インタラクティブな計算と
無関係な Web サービスの提供、および無料枠での「ノートブック UI を迂回して Web UI を
主眼として操作すること」とリモートプロキシへの接続を禁じています。トンネルは VM の
リセットで失われ、アイドル状態が続くとランタイムごと終了します。

公開 URL が必要な場合は、同じアプリを Cloudflare Tunnel か Vercel に deploy してください。
このノートブックはサンドボックス側の実行基盤としてだけ使います。

**このノートブックで動作するのは demo backend のみです。** Colab には Docker daemon も
OpenShell gateway もないため、実 sandbox は起動しません。画面上の操作は本物ですが、
応答データは擬似値です。

In [ ]:
import subprocess
import sys

REPO = "https://github.com/watanabe3tipapa/openshell-plus"
PORT = 8080

# このリポジトリが public かつ push 済みでないとインストールできない。
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", f"git+{REPO}"],
    check=False,
)
if result.returncode != 0:
    raise SystemExit(
        f"pip install failed for {REPO} "
        f"（{REPO} の pip install に失敗しました。"
        "このリポジトリが public かつ push 済みである必要があります。）"
    )
print(f"installed openshell-ui from GitHub（{REPO} から導入しました）")

## 起動モード

Colab には Docker daemon も OpenShell gateway もないため、アプリは demo モードで起動します。
自動判定に任せるのではなく `OSUI_DEMO=true` を明示的に設定して、demo で動くことを
はっきりさせています。

次のセルはポート `127.0.0.1:8080` が接続を受けるまで最大 30 秒待ちます。
30 秒を超えると `RuntimeError` で停止します。

In [ ]:
import os

os.environ["OSUI_DEMO"] = "true"
os.environ["OSUI_MODE"] = "colab"
os.environ["OSUI_HOST"] = "127.0.0.1"
os.environ["OSUI_PORT"] = str(PORT)

server = subprocess.Popen(
    ["uvicorn", "openshell_ui.main:app", "--host", "127.0.0.1", "--port", str(PORT)],
    env={**os.environ},
)
print(f"uvicorn pid={server.pid} on 127.0.0.1:{PORT}（uvicorn を起動しました）")

In [ ]:
import socket
import time

for _ in range(60):
    with socket.socket() as sock:
        if sock.connect_ex(("127.0.0.1", PORT)) == 0:
            print("dashboard is accepting connections")
            break
    time.sleep(0.5)
else:
    raise RuntimeError("dashboard did not start")

## ノートブック内に表示する

`serve_kernel_port_as_iframe` は `google.colab.output._util` の内部ヘルパであり、
公式にサポートされた公開 API ではありません。Colab はポート転送の公開 API を
提供しておらず、ポートを公開することを推奨する公式ドキュメントも存在しません。

下のセルは iframe でダッシュボードを埋め込み、ノートブックの UI を主役に保ちます。
内部ヘルパが利用できなくなった場合は、公式に export されている
`serve_kernel_port_as_window` にフォールバックします。こちらは別のブラウザタブで開きます。

In [ ]:
# 内部の iframe ヘルパを優先し、ノートブックの UI を主役に保つ。
# 存在しない場合は、公式の window ヘルパ（新しいタブで開く）へフォールバックする。
try:
    from google.colab.output._util import serve_kernel_port_as_iframe
except ImportError:
    from google.colab.output import serve_kernel_port_as_window

    serve_kernel_port_as_window(PORT, path="/")
else:
    serve_kernel_port_as_iframe(PORT, path="/", width="100%", height="760")

## サポートしません：公開トンネル

次のセルは意図的に無効のままです。コメントアウトを解除すると、認証のないランタイムを
インターネットへ公開することになり、Colab の利用規約に抵触します。
その URL は VM とともに消えます。ローカルの環境では
`scripts/run-cloudflare.sh quick` を使ってください。

In [ ]:
# subprocess.run(["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"], check=False)